# 05 – N-BEATS tipo modelio tyrimas ir tobulinimas

## Darbo tikslas

Šiame Notebook tiriama lengva **N-BEATS tipo bendrosios architektūros realizacija**, naudojanti `backcast` ir `forecast` blokus. Ji nėra pristatoma kaip identiška oficialiai N-BEATS realizacijai.

Sprendžiamas tas pats uždavinys kaip kituose projekto modeliuose – pagal iki prognozavimo momento prieinamą informaciją prognozuojama **kitos valandos dviračių nuomos paklausa**.

Pagrindinė 24 valandų sekos įvestis:

$$
X_t=[y_{t-24},y_{t-23},\ldots,y_{t-1}]
$$

o modelio išvestis:

$$
\hat y_t
$$

### Modelio atrankos principas

N-BEATS hiperparametrai parenkami **tik DEV dalyje pagal mažiausią vidutinę MAE** per tuos pačius chronologinius `rolling-origin` skaidinius. RMSE ir Peak MAE naudojamos papildomam prognozės klaidų vertinimui.

**PUR (Peak Underestimation Rate)** šiame Notebook paliekamas kaip papildomas diagnostinis piko elgsenos rodiklis. Jis parodo, kokia piko valandų dalis buvo neįvertinta, tačiau **pagal PUR modelio konfigūracija neparenkama**.

Toliau atliekama 24 ir 168 valandų retrospektyvaus lango abliacija, 5 % trūkstamų įvesties reikšmių atsparumo bandymas ir prognozavimo klaidų analizė. **TEST šiame Notebook modelio parinkimui nenaudojamas.**

In [ ]:
from pathlib import Path
import sys
import warnings
import json

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_PATH = ROOT / "hour.csv"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

from src.common import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Fiksuota sėkla – rezultatų atkuriamumui.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Bendras duomenų paruošimas toks pats kaip kituose projekto Notebook.
work = load_prepare(DATA_PATH)
dev, test = split_dev_test(work)
folds = rolling_origin_folds(dev)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 1. Eksperimento korektiškumas ir sekų duomenų nutekėjimo kontrolė

Sekų modeliui neužtenka vien chronologiškai atskirti DEV ir TEST. Kiekvienai prognozei taip pat būtina užtikrinti, kad į įvesties seką nepatektų prognozuojamos valandos ar dar vėlesnė tikroji paklausa.

Todėl prognozuojant paklausą laiku $t$, įvesties seka sudaroma tik iš ankstesnių laikų:

$$
t-L,\ldots,t-2,t-1
$$

čia $L$ – retrospektyvaus lango ilgis.

Modelio hiperparametrai parenkami tik DEV dalyje, naudojant chronologinius `rolling-origin` skaidinius. TEST paliekamas galutiniam bendram modelių palyginimui.

In [ ]:
# Parodoma, kiek duomenų palikta modelių kūrimui (DEV) ir kiek – galutiniam TEST.
# Šiame Notebook TEST tik parodomas, bet modelio parinkimui nenaudojamas.
print("DEV eilučių skaičius:", len(dev))
print("TEST eilučių skaičius:", len(test))
print("Chronologinių DEV skaidinių skaičius:", len(folds))
print("TEST šiame Notebook modelio parinkimui nenaudojamas.")

## 2. Sekų sudarymas

Funkcija `sequences()` kiekvienam tiksliniam laikui surenka tiksliai $L$ ankstesnių valandų tikrąsias paklausos reikšmes. Jeigu bent vienos reikiamos valandos nėra, toks pavyzdys nesudaromas.

Tokiu būdu sekos remiasi **tiksliais laiko žingsniais**, o ne vien gretimų lentelės eilučių pozicijomis.

In [ ]:
# Funkcija kiekvienai prognozuojamai valandai sudaro tik iš ankstesnės
# paklausos reikšmių sudarytą seką.
def sequences(frame, lookback=24):
    # target_time – valanda, kurios paklausa yra stulpelyje target.
    # Istorija indeksuojama pagal tikslų laiką, todėl nenaudojamos vien
    # gretimos lentelės eilutės.
    yhist = frame.set_index("target_time")["target"].sort_index()

    X = []
    y = []
    times = []

    for _, row in frame.iterrows():
        t = pd.Timestamp(row["target_time"])
        vals = []
        complete = True

        # Surenkamos tik ankstesnės valandos:
        # t-lookback, ..., t-2, t-1.
        for h in range(lookback, 0, -1):
            tt = t - pd.Timedelta(hours=h)

            # Jei bent vienos reikiamos valandos nėra, nepilna seka praleidžiama.
            if tt not in yhist.index:
                complete = False
                break

            vals.append(float(yhist.loc[tt]))

        if complete:
            X.append(vals)
            y.append(float(row["target"]))
            times.append(t)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        pd.to_datetime(times)
    )


# Greita funkcijos patikra su 24 val. retrospektyviu langu.
Xdev, ydev, tdev = sequences(dev, 24)

display(pd.DataFrame({"Retrospektyvus langas (val.)": [Xdev.shape[1]], "Sudarytų sekų skaičius": [len(Xdev)]}))

assert len(Xdev) == len(ydev) == len(tdev)
assert Xdev.ndim == 2 and Xdev.shape[1] == 24

print("SEKŲ PATIKRA PRAEITA: kiekvieną įvestį sudaro 24 ankstesnės valandos.")


In [ ]:
# Iš DEV duomenų sudaromos 24 ankstesnių valandų paklausos sekos.
# Xdev – įvesties sekos, ydev – tikrosios prognozuojamos reikšmės,
# tdev – prognozuojamų valandų laiko žymės.
Xdev, ydev, tdev = sequences(dev, 24)

assert len(tdev) == len(Xdev)
assert Xdev.shape[1] == 24
assert pd.Series(tdev).is_monotonic_increasing

print("24 val. DEV sekų skaičius:", len(Xdev))
print("Vienos sekos ilgis:", Xdev.shape[1])
print("SEKŲ PATIKRA PRAEITA: įvesties seka sudaryta iš 24 ankstesnių valandų.")
print("Prognozuojamos valandos target reikšmė į tos pačios prognozės įvestį neįtraukiama.")

## 3. N-BEATS tipo bloko veikimo principas

Kiekvienas blokas iš esamos įvesties apskaičiuoja:

- **backcast** $\hat{x}^{(b)}$ – bloko paaiškinamą įvesties dalį;
- **forecast** $\hat{y}^{(b)}$ – bloko indėlį į prognozę.

Po kiekvieno bloko atnaujinama liekana:

$$
x^{(b+1)}
=
x^{(b)}-\hat{x}^{(b)}
$$

o galutinė prognozė gaunama sudedant visų blokų `forecast` indėlius:

$$
\hat{y}
=
\sum_{b=1}^{B}\hat{y}^{(b)}
$$

Kode tai tiesiogiai atitinka:

```python
residual = residual - back
forecast = forecast + fore
```

Todėl formulę galima tiesiogiai susieti su `NBEATS.forward()` realizacija.

In [ ]:
class Block(nn.Module):
    # Vienas blokas iš įvesties apskaičiuoja backcast ir forecast.
    def __init__(self, backcast_len=24, hidden=128):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(backcast_len, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU()
        )

        self.back = nn.Linear(hidden, backcast_len)
        self.fore = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.fc(x)
        backcast = self.back(h)
        forecast = self.fore(h)
        return backcast, forecast


class NBEATS(nn.Module):
    # Lengva N-BEATS tipo architektūra.
    # Kiekvieno bloko backcast atimamas iš liekanos,
    # o forecast indėliai sudedami į galutinę prognozę.
    def __init__(self, backcast_len=24, hidden=128, blocks=3):
        super().__init__()

        self.blocks = nn.ModuleList(
            [Block(backcast_len, hidden) for _ in range(blocks)]
        )

    def forward(self, x):
        residual = x
        forecast = torch.zeros(
            x.shape[0], 1,
            dtype=x.dtype,
            device=x.device
        )

        for block in self.blocks:
            back, fore = block(residual)
            residual = residual - back
            forecast = forecast + fore

        return forecast.squeeze(-1)


# Greita architektūros patikra prieš pradedant ilgą hiperparametrų paiešką.
_test_model = NBEATS(backcast_len=24, hidden=64, blocks=2).to(DEVICE)
_test_x = torch.zeros((2, 24), dtype=torch.float32, device=DEVICE)
_test_y = _test_model(_test_x)

assert _test_y.shape == (2,)
del _test_model, _test_x, _test_y


## 4. Modelio mokymas

Mokymo metu kiekvieno `rolling-origin` TRAIN laikotarpio paskutiniai 15 % sekų naudojami **vidiniam chronologiniam early stopping** vertinimui. Duomenys nėra maišomi.

Įvesties ir tikslo mastelio parametrai apskaičiuojami tik iš vidinės mokymo dalies. Modelio būsena išsaugoma tada, kai gerėja vidinės validavimo dalies MAE.

Nuostolio funkcija:

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum_{i=1}^{N}
|y_i-\hat y_i|
$$

Kode ją atitinka `nn.L1Loss()`.

In [ ]:
# Apmoko vieną N-BEATS tipo modelį ir grąžina VALIDATION prognozes.
def fit_predict(
    Xtr, ytr, Xv,
    hidden=128,
    blocks=3,
    lr=1e-3,
    epochs=80,
    patience=10,
    batch_size=256,
    return_history=False
):
    if len(Xtr) < 2:
        raise ValueError("TRAIN sekų per mažai modelio mokymui.")
    if len(Xv) == 0:
        raise ValueError("VALIDATION sekų nėra.")

    # Vienoda pradžia kiekvienam eksperimentui – atkuriamumui.
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # Paskutiniai 15 % TRAIN sekų naudojami chronologiniam early stopping.
    cut = int(len(Xtr) * 0.85)
    cut = min(max(cut, 1), len(Xtr) - 1)

    Xfit, yfit = Xtr[:cut], ytr[:cut]
    Xes, yes = Xtr[cut:], ytr[cut:]

    # Mastelio parametrai skaičiuojami tik iš vidinės TRAIN dalies.
    mu = float(Xfit.mean())
    sd = float(Xfit.std()) + 1e-8
    ym = float(yfit.mean())
    ys = float(yfit.std()) + 1e-8

    A = torch.tensor((Xfit - mu) / sd, dtype=torch.float32)
    B = torch.tensor((yfit - ym) / ys, dtype=torch.float32)
    E = torch.tensor((Xes - mu) / sd, dtype=torch.float32)
    Ey = torch.tensor((yes - ym) / ys, dtype=torch.float32)
    V = torch.tensor((Xv - mu) / sd, dtype=torch.float32)

    model = NBEATS(
        backcast_len=Xtr.shape[1],
        hidden=hidden,
        blocks=blocks
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()

    loader = DataLoader(
        TensorDataset(A, B),
        batch_size=batch_size,
        shuffle=False
    )

    best_val = np.inf
    wait = 0
    best_state = None
    train_hist = []
    val_hist = []

    for epoch in range(epochs):
        model.train()
        batch_losses = []

        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

            batch_losses.append(float(loss.item()))

        model.eval()
        with torch.no_grad():
            val_loss = float(
                loss_fn(
                    model(E.to(DEVICE)),
                    Ey.to(DEVICE)
                ).item()
            )

        train_loss = float(np.mean(batch_losses))
        train_hist.append(train_loss)
        val_hist.append(val_loss)

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            wait = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            wait += 1

        if wait >= patience:
            break

    if best_state is None:
        raise RuntimeError("Nepavyko išsaugoti geriausios modelio būsenos.")

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred = (
            model(V.to(DEVICE)).cpu().numpy() * ys + ym
        )

    used_epochs = epoch + 1

    if return_history:
        return pred, used_epochs, {
            "train": train_hist,
            "val": val_hist
        }

    return pred, used_epochs


### Modelio įgyvendinimo patikra

Prieš eksperimentus programiškai patikrinama, ar sekų sudarymo funkcija, N-BEATS tipo modelis ir mokymo funkcija yra apibrėžtos, o `rolling-origin` skaidiniuose galima sudaryti 24 val. įvesties sekas. Šios patikros paskirtis – užtikrinti, kad tolesni eksperimentai būtų vykdomi su nuosekliai paruoštais duomenimis.

In [ ]:
# Ši celė turi praeiti be klaidų prieš pradedant hiperparametrų paiešką.
assert callable(sequences)
assert "NBEATS" in globals()
assert callable(fit_predict)
assert len(folds) > 0

# Trumpas vieno skaidinio duomenų patikrinimas.
_tri, _vai = folds[0]
_tr = dev.iloc[_tri]
_va = dev.iloc[_vai]

_Xtr_check, _ytr_check, _ = sequences(_tr, 24)
_Xv_check, _yv_check, _ = sequences(_va, 24)

assert len(_Xtr_check) > 0
assert len(_Xv_check) > 0
assert _Xtr_check.shape[1] == 24
assert _Xv_check.shape[1] == 24


del _tri, _vai, _tr, _va
del _Xtr_check, _ytr_check, _Xv_check, _yv_check


## 5. Hiperparametrų paieška DEV

Kad N-BEATS būtų vertinamas tokiomis pačiomis principinėmis sąlygomis kaip kiti projekto modeliai, konfigūracija parenkama pagal **vidutinę DEV MAE** visuose 3 chronologiniuose `rolling-origin` skaidiniuose.

Tikrinami:

- paslėpto sluoksnio neuronų skaičius `hidden`;
- N-BEATS tipo blokų skaičius `blocks`;
- mokymosi sparta `lr`.

Paieška sąmoningai ribota dėl neuroninio modelio mokymo trukmės, tačiau kiekviena konfigūracija vertinama vienodomis DEV sąlygomis. TEST neliečiamas.

**Atrankos taisyklė:** geriausia konfigūracija nustatoma pagal mažiausią vidutinę DEV MAE. PUR į atrankos kriterijų neįtraukiamas; jis pateikiamas tik papildomai piko elgsenai aprašyti.

In [ ]:
# Ribotas hiperparametrų rinkinys. Visos konfigūracijos bus vertinamos
# tomis pačiomis trimis chronologinėmis DEV validavimo dalimis.
configs = [
    {"hidden": 64,  "blocks": 2, "lr": 1e-3},
    {"hidden": 128, "blocks": 2, "lr": 1e-3},
    {"hidden": 128, "blocks": 3, "lr": 1e-3},
    {"hidden": 128, "blocks": 4, "lr": 5e-4},
    {"hidden": 256, "blocks": 3, "lr": 5e-4},
    {"hidden": 256, "blocks": 4, "lr": 5e-4},
]

rows = []

# Kiekviena konfigūracija išbandoma visuose rolling-origin skaidiniuose.
# Galutinis jos įvertis – metrikų vidurkis per visus skaidinius.
for i, cfg in enumerate(configs, 1):
    print(f"N-BEATS konfigūracija {i}/{len(configs)}: {cfg}")
    vals = []

    for fold_id, (tri, vai) in enumerate(folds, 1):
        # TRAIN visada yra ankstesnis laikotarpis, VALIDATION – vėlesnis.
        tr, va = dev.iloc[tri], dev.iloc[vai]
        Xtr, ytr, _ = sequences(tr, 24)
        Xv, yv, _ = sequences(va, 24)

        pred, ep = fit_predict(Xtr, ytr, Xv, **cfg)

        r = metrics(
            yv, pred,
            float(tr["target"].quantile(PEAK_Q))
        )
        vals.append({"fold": fold_id, "epochs": ep, **r})

    fold_df = pd.DataFrame(vals)
    mean_r = fold_df[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()

    rows.append({
        **cfg,
        **mean_r.to_dict(),
        "Mean_epochs": fold_df["epochs"].mean()
    })

nr = pd.DataFrame(rows).sort_values(["MAE", "RMSE"]).reset_index(drop=True)
display(nr)

# Pasirenkama mažiausią vidutinę DEV MAE gavusi konfigūracija.
# TEST rezultatas šiam pasirinkimui nenaudojamas.
best_n = {
    "hidden": int(nr.iloc[0]["hidden"]),
    "blocks": int(nr.iloc[0]["blocks"]),
    "lr": float(nr.iloc[0]["lr"])
}

print("Geriausia DEV konfigūracija:", best_n)
print(f"Geriausia vidutinė DEV MAE: {nr.iloc[0]['MAE']:.3f}")

plt.figure(figsize=(10, 4))
labels = [
    f"H{int(r.hidden)} B{int(r.blocks)} LR{r.lr:g}"
    for _, r in nr.iterrows()
]
plt.bar(labels, nr["MAE"])
plt.ylabel("DEV MAE")
plt.title("N-BEATS konfigūracijų palyginimas")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 6. N-BEATS modelio patobulinimo įvertinimas

Kad modelio tobulinimas būtų aiškiai matomas, atskirai palyginamas:

- **pradinis N-BEATS tipo modelis** – pirmoji iš anksto nustatyta konfigūracija: `hidden=64`, `blocks=2`, `lr=0.001`;
- **patobulintas N-BEATS tipo modelis** – iš visų patikrintų konfigūracijų mažiausią vidutinę DEV MAE pasiekęs variantas.

Abu variantai vertinami **tais pačiais 3 chronologiniais DEV `rolling-origin` skaidiniais**. TEST nenaudojamas.

Pagerėjimas apskaičiuojamas:

$$
\mathrm{Pagerėjimas}
=
\frac{
\mathrm{MAE}_{pradinis}
-
\mathrm{MAE}_{patobulintas}
}{
\mathrm{MAE}_{pradinis}
}
\cdot100\%
$$

Teigiama reikšmė reiškia, kad hiperparametrų paieška sumažino DEV MAE. Jeigu reikšmė lygi nuliui, pradinė konfigūracija jau buvo geriausia iš patikrintų. Rezultatas nėra iš anksto laikomas pagerėjimu – tai turi parodyti DEV metrikos.

In [ ]:
# Pirmoji iš anksto nustatyta konfigūracija laikoma pradiniu modeliu.
initial_cfg = configs[0]

# Randame jos jau apskaičiuotus rezultatus iš bendros paieškos lentelės.
initial_row = nr[
    (nr["hidden"] == initial_cfg["hidden"]) &
    (nr["blocks"] == initial_cfg["blocks"]) &
    (np.isclose(nr["lr"], initial_cfg["lr"]))
].iloc[0]

# Geriausias paieškos rezultatas yra pirmoje surikiuotos lentelės eilutėje.
improved_row = nr.iloc[0]

improvement_table = pd.DataFrame({
    "Modelis": ["Pradinis N-BEATS", "Patobulintas N-BEATS"],
    "hidden": [int(initial_row["hidden"]), int(improved_row["hidden"])],
    "blocks": [int(initial_row["blocks"]), int(improved_row["blocks"])],
    "lr": [float(initial_row["lr"]), float(improved_row["lr"])],
    "MAE": [float(initial_row["MAE"]), float(improved_row["MAE"])],
    "RMSE": [float(initial_row["RMSE"]), float(improved_row["RMSE"])],
    "Peak_MAE": [float(initial_row["Peak_MAE"]), float(improved_row["Peak_MAE"])],
    "PUR": [float(initial_row["PUR"]), float(improved_row["PUR"])]
}).set_index("Modelis")

display(improvement_table)

initial_mae = float(initial_row["MAE"])
improved_mae = float(improved_row["MAE"])
improvement_pct = (initial_mae - improved_mae) / initial_mae * 100

if improvement_pct > 1e-9:
    improvement_text = (
        f"Hiperparametrų paieška DEV MAE sumažino nuo "
        f"{initial_mae:.3f} iki {improved_mae:.3f}, "
        f"t. y. {improvement_pct:.2f} %."
    )
else:
    improvement_text = (
        "Papildoma hiperparametrų paieška DEV MAE nepagerino: "
        "pradinė konfigūracija liko geriausia iš patikrintų variantų."
    )

display(Markdown("### Patobulinimo rezultatas\n\n" + improvement_text))

# PUR pokytis parodomas tik kaip papildoma diagnostika.
initial_pur = float(initial_row["PUR"])
improved_pur = float(improved_row["PUR"])
pur_diff = improved_pur - initial_pur

if pur_diff > 1e-9:
    pur_text = (
        f"Papildomai nustatyta, kad PUR padidėjo nuo {initial_pur:.3f} "
        f"iki {improved_pur:.3f}. Tai rodo, kad pagal piko neįvertinimo "
        f"dažnį patobulintas variantas veikė prasčiau. Vis dėlto PUR nėra "
        f"šio modelio atrankos kriterijus, todėl galutinė N-BEATS konfigūracija "
        f"ir toliau pasirenkama pagal mažesnę DEV MAE."
    )
elif pur_diff < -1e-9:
    pur_text = (
        f"Papildomai PUR sumažėjo nuo {initial_pur:.3f} iki {improved_pur:.3f}. "
        f"Tai yra palankus piko elgsenos rezultatas, tačiau PUR modelio "
        f"konfigūracijos atrankai nenaudojamas."
    )
else:
    pur_text = (
        "Papildomas PUR rodiklis nepasikeitė. Jis pateikiamas piko elgsenos "
        "diagnostikai, bet modelio konfigūracijos atrankai nenaudojamas."
    )

display(Markdown("### Papildoma piko elgsenos pastaba\n\n" + pur_text))


### Pradinio ir patobulinto modelio grafinis palyginimas

MAE, RMSE ir Peak MAE matuojamos dviračių nuomų skaičiumi, todėl jos pateikiamos viename grafike. PUR rodoma atskirai dėl 0–1 skalės.

Svarbu: **PUR grafikas yra papildoma piko elgsenos diagnostika, o ne modelio atrankos kriterijus.** Galutinė N-BEATS konfigūracija pasirenkama pagal DEV MAE.

In [ ]:
# Bendrų ir piko klaidų palyginimas.
ax = improvement_table[["MAE", "RMSE", "Peak_MAE"]].plot(
    kind="bar", figsize=(10, 5)
)
ax.set_ylabel("Klaida, nuomų skaičius")
ax.set_title("Pradinis ir patobulintas N-BEATS tipo modelis")
ax.set_xticklabels(
    ["Pradinis N-BEATS", "Patobulintas N-BEATS"],
    rotation=0
)
ax.legend(["MAE", "RMSE", "Peak MAE"])
plt.tight_layout()
plt.show()

# PUR pateikiamas atskirai, kad skirtingi metrikų masteliai neklaidintų.
plt.figure(figsize=(7, 4))
plt.bar(
    ["Pradinis N-BEATS", "Patobulintas N-BEATS"],
    improvement_table["PUR"]
)
plt.ylabel("PUR")
plt.ylim(0, 1)
plt.title("Pradinio ir patobulinto N-BEATS PUR palyginimas")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Ką šis eksperimentas parodo?

Ši dalis parodo, ar N-BEATS hiperparametrų keitimas sumažina pagrindinę modelio atrankos metriką – **DEV MAE**. Patobulintu laikomas tas iš patikrintų variantų, kuris pasiekia mažiausią vidutinę MAE per chronologinius `rolling-origin` skaidinius.

RMSE ir Peak MAE padeda papildomai įvertinti didesnes bei piko laikotarpių klaidas. PUR taip pat pateikiamas, tačiau naudojamas tik kaip papildomas piko neįvertinimo diagnostinis rodiklis. Todėl didesnis PUR savaime nekeičia pagal MAE pasirinktos N-BEATS konfigūracijos, tačiau toks pablogėjimas turi būti aiškiai nurodomas kaip modelio ribotumas.

Toliau atliekama abliacija atsako į kitą klausimą – kaip rezultatą keičia retrospektyvaus istorijos lango ilgis, kai modelio hiperparametrai jau užfiksuoti.

## 7. Pasirinktos konfigūracijos stabilumas DEV skaidiniuose

Geriausia konfigūracija dar kartą vertinama kiekviename DEV skaidinyje atskirai. Taip matyti, ar bendras vidurkis nėra nulemtas tik vieno laikotarpio.

In [ ]:
# Pakartotinai įvertiname pasirinktą konfigūraciją kiekviename skaidinyje,
# kad matytume ne tik vidurkį, bet ir rezultato stabilumą laike.
best_fold_rows = []

for fold_id, (tri, vai) in enumerate(folds, 1):
    tr, va = dev.iloc[tri], dev.iloc[vai]
    Xtr, ytr, _ = sequences(tr, 24)
    Xv, yv, _ = sequences(va, 24)

    pred, ep = fit_predict(Xtr, ytr, Xv, **best_n)

    best_fold_rows.append({
        "fold": fold_id,
        "epochs": ep,
        **metrics(yv, pred, float(tr["target"].quantile(PEAK_Q)))
    })

best_folds = pd.DataFrame(best_fold_rows)
display(best_folds)

best_mean = best_folds[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()
display(pd.DataFrame([best_mean], index=["DEV vidurkis"]))

### Rezultato stabilumas skirtinguose DEV laikotarpiuose

Grafikas parodo pasirinkto modelio MAE kiekviename chronologiniame DEV skaidinyje. Kuo mažesnis skirtumas tarp skaidinių, tuo stabilesnis modelio rezultatas skirtingais laiko laikotarpiais.

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(best_folds["fold"].astype(str), best_folds["MAE"])
plt.xlabel("DEV rolling-origin skaidinys")
plt.ylabel("MAE")
plt.title("N-BEATS MAE stabilumas DEV skaidiniuose")
plt.tight_layout()
plt.show()

## 8. Palyginimas su sezoniniu atskaitos metodu

N-BEATS DEV rezultatas palyginamas su tuo pačiu sezoniniu atskaitos metodu, kuris naudojamas kituose projekto Notebook.

Svarbu: N-BEATS prognozėms metrikos skaičiuojamos tik toms validavimo valandoms, kurioms galima sudaryti pilną 24 valandų seką. Todėl baseline prognozė šiame palyginime sulygiuojama pagal tas pačias N-BEATS vertinimo laiko žymes.

In [ ]:
# Tas pats sezoninis baseline vertinamas tomis pačiomis laiko žymėmis,
# kurioms N-BEATS gali sudaryti pilną įvesties seką.
baseline_rows = []

for fold_id, (tri, vai) in enumerate(folds, 1):
    tr, va = dev.iloc[tri], dev.iloc[vai]
    Xv, yv, tv = sequences(va, 24)

    base_all = seasonal_baseline(tr, va, k_weeks=4)
    base_series = pd.Series(
        np.asarray(base_all),
        index=pd.to_datetime(va["target_time"]).to_numpy()
    )
    base_pred = base_series.loc[pd.to_datetime(tv)].to_numpy()

    baseline_rows.append({
        "fold": fold_id,
        **metrics(yv, base_pred, float(tr["target"].quantile(PEAK_Q)))
    })

baseline_folds = pd.DataFrame(baseline_rows)
baseline_mean = baseline_folds[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()

comparison = pd.DataFrame({
    "Sezoninis atskaitos metodas": baseline_mean,
    "N-BEATS tipo modelis": best_mean
}).T
display(comparison)

mae_change = (
    (baseline_mean["MAE"] - best_mean["MAE"])
    / baseline_mean["MAE"] * 100
)

if mae_change > 0:
    txt = f"N-BEATS DEV MAE yra {mae_change:.1f} % mažesnė už sezoninio atskaitos metodo MAE."
else:
    txt = f"N-BEATS DEV MAE yra {abs(mae_change):.1f} % didesnė už sezoninio atskaitos metodo MAE."

print(txt)

### Grafinis N-BEATS ir atskaitos metodo palyginimas

Kad palyginimas būtų vienodas su kitais projekto modeliais, atskirai vizualizuojamos visos keturios vertinimo metrikos. MAE, RMSE ir Peak MAE išreiškiamos nuomų skaičiumi, todėl jos rodomos viename grafike. PUR yra dalis nuo 0 iki 1, todėl pateikiama atskirai, kad skirtingi masteliai neklaidintų.

In [ ]:
# Klaidos metrikos turi tą patį matavimo vienetą – nuomų skaičių.
error_metrics = ["MAE", "RMSE", "Peak_MAE"]
plot_compare = comparison[error_metrics].copy()

ax = plot_compare.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Klaida, nuomų skaičius")
ax.set_title("N-BEATS ir sezoninio atskaitos metodo klaidų palyginimas")
ax.set_xticklabels(["Sezoninis atskaitos metodas", "N-BEATS tipo modelis"], rotation=0)
ax.legend(["MAE", "RMSE", "Peak MAE"])
plt.tight_layout()
plt.show()

# PUR rodomas atskirai, nes jo skalė 0–1.
plt.figure(figsize=(7, 4))
plt.bar(["Sezoninis atskaitos metodas", "N-BEATS tipo modelis"],
        comparison["PUR"])
plt.ylabel("PUR")
plt.ylim(0, 1)
plt.title("Piko paklausos neįvertinimo dalis (PUR)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 9. Abliacijos bandymas – 24 val. ir 168 val. retrospektyvus langas

Dabartinė N-BEATS tipo realizacija naudoja tik ankstesnės paklausos seką, todėl meteorologinių požymių abliacija šiam modeliui nebūtų pagrįsta.

Tikrinama realiai modelyje naudojamos įvesties savybė – **istorijos lango ilgis**:

- 24 val. – ankstesnė para;
- 168 val. – ankstesnė savaitė.

Kad palyginimas būtų korektiškas, abu variantai vertinami tais pačiais DEV `rolling-origin` skaidiniais. Kadangi 168 val. langui galima sudaryti mažiau sekų, metrikos kiekviename skaidinyje skaičiuojamos pagal **bendras abiejų variantų laiko žymes**.

In [ ]:
# ABLIACIJA: keičiama tik viena modelio įvesties savybė – istorijos lango ilgis.
# Kiti pasirinktos konfigūracijos parametrai paliekami tokie patys.
ablation_rows = []

for fold_id, (tri, vai) in enumerate(folds, 1):
    tr, va = dev.iloc[tri], dev.iloc[vai]

    # Paruošiami abu langai.
    prepared = {}
    for L in [24, 168]:
        Xtr, ytr, _ = sequences(tr, L)
        Xv, yv, tv = sequences(va, L)

        pred, ep = fit_predict(
            Xtr, ytr, Xv,
            hidden=best_n["hidden"],
            blocks=best_n["blocks"],
            lr=best_n["lr"]
        )
        prepared[L] = pd.DataFrame({
            "target_time": pd.to_datetime(tv),
            "y": yv,
            "pred": pred
        })

    # 24 h ir 168 h variantai gali turėti skirtingą galimų sekų skaičių.
    # Todėl sąžiningam palyginimui paliekamos tik bendros laiko žymės.
    common_times = (
        set(prepared[24]["target_time"])
        & set(prepared[168]["target_time"])
    )

    for L in [24, 168]:
        z = prepared[L][prepared[L]["target_time"].isin(common_times)].copy()
        z = z.sort_values("target_time")
        r = metrics(
            z["y"].to_numpy(),
            z["pred"].to_numpy(),
            float(tr["target"].quantile(PEAK_Q))
        )
        ablation_rows.append({
            "fold": fold_id,
            "Retrospektyvus_langas_val": L,
            **r
        })

ablation_folds = pd.DataFrame(ablation_rows)
display(ablation_folds)

ablation = (
    ablation_folds
    .groupby("Retrospektyvus_langas_val")[["MAE", "RMSE", "Peak_MAE", "PUR"]]
    .mean()
    .reset_index()
)
display(ablation)

plt.figure(figsize=(8, 4))
plt.bar(
    ablation["Retrospektyvus_langas_val"].astype(str),
    ablation["MAE"]
)
plt.xlabel("Retrospektyvus langas, val.")
plt.ylabel("DEV MAE")
plt.title("N-BEATS abliacija: istorijos lango įtaka")
plt.tight_layout()
plt.show()

best_window = int(
    ablation.loc[ablation["MAE"].idxmin(), "Retrospektyvus_langas_val"]
)
print("Mažesnę vidutinę DEV MAE gavo", best_window, "val. langas.")

### Išsamesnis abliacijos rezultatų palyginimas

Vien MAE neparodo viso istorijos lango pakeitimo poveikio. Todėl papildomai grafiškai palyginamos RMSE, Peak MAE ir PUR. Taip galima matyti, ar vienas retrospektyvus langas pagerina bendrą tikslumą, bet kartu nepablogina prognozių piko metu.

In [ ]:
# MAE, RMSE ir Peak MAE – vienodo vieneto klaidų metrikos.
abl_plot = ablation.set_index("Retrospektyvus_langas_val")
ax = abl_plot[["MAE", "RMSE", "Peak_MAE"]].plot(
    kind="bar", figsize=(9, 5)
)
ax.set_xlabel("Retrospektyvus langas, val.")
ax.set_ylabel("Klaida, nuomų skaičius")
ax.set_title("Abliacija: 24 h ir 168 h langų klaidų metrikos")
ax.legend(["MAE", "RMSE", "Peak MAE"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# PUR dėl kito mastelio rodomas atskirai.
plt.figure(figsize=(7, 4))
plt.bar(
    abl_plot.index.astype(str),
    abl_plot["PUR"]
)
plt.xlabel("Retrospektyvus langas, val.")
plt.ylabel("PUR")
plt.ylim(0, 1)
plt.title("Abliacija: 24 h ir 168 h langų PUR")
plt.tight_layout()
plt.show()

## 10. Atsparumo bandymas – 5 % trūkstamų įvesties reikšmių

Atsparumo bandyme kiekvieno DEV skaidinio validavimo įvestyse atsitiktinai pašalinama 5 % reikšmių. Naudojama fiksuota `SEED`, todėl bandymas atkuriamas.

Trūkstamos reikšmės užpildomos **tik atitinkamo TRAIN sekų mediana**. Tas pats apmokytas modelis turi būti vertinamas su švaria ir pažeista validavimo įvestimi, todėl atsparumo palyginime modelis nėra mokomas iš naujo vien dėl to, kad pakeista validavimo įvestis.

In [ ]:
# Atsparumo bandymas: vienas modelis apmokomas su TRAIN ir tada tas pats
# modelis vertinamas su švaria bei 5 % pažeista VALIDATION įvestimi.
def fit_predict_clean_and_missing(
    Xtr, ytr, Xv,
    hidden, blocks, lr,
    missing_rate=0.05,
    epochs=80,
    patience=10,
    batch_size=256
):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    cut = int(len(Xtr) * 0.85)
    cut = min(max(cut, 1), len(Xtr) - 1)

    Xfit, yfit = Xtr[:cut], ytr[:cut]
    Xes, yes = Xtr[cut:], ytr[cut:]

    mu = float(Xfit.mean())
    sd = float(Xfit.std()) + 1e-8
    ym = float(yfit.mean())
    ys = float(yfit.std()) + 1e-8

    A = torch.tensor((Xfit - mu) / sd, dtype=torch.float32)
    B = torch.tensor((yfit - ym) / ys, dtype=torch.float32)
    E = torch.tensor((Xes - mu) / sd, dtype=torch.float32)
    Ey = torch.tensor((yes - ym) / ys, dtype=torch.float32)

    model = NBEATS(
        backcast_len=Xtr.shape[1],
        hidden=hidden,
        blocks=blocks
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()

    loader = DataLoader(
        TensorDataset(A, B),
        batch_size=batch_size,
        shuffle=False
    )

    best_val = np.inf
    wait = 0
    best_state = None

    for epoch in range(epochs):
        model.train()

        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = float(
                loss_fn(
                    model(E.to(DEVICE)),
                    Ey.to(DEVICE)
                ).item()
            )

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            wait = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            wait += 1

        if wait >= patience:
            break

    if best_state is None:
        raise RuntimeError("Nepavyko išsaugoti geriausios modelio būsenos.")

    model.load_state_dict(best_state)
    model.eval()

    # 1) Švari VALIDATION įvestis.
    V_clean = torch.tensor(
        (Xv - mu) / sd,
        dtype=torch.float32
    )

    # 2) Ta pati VALIDATION įvestis, bet atsitiktinai pašalinama 5 % reikšmių.
    rng = np.random.default_rng(SEED)
    X_missing = Xv.copy()

    mask = rng.random(X_missing.shape) < missing_rate
    X_missing[mask] = np.nan

    # Užpildymui naudojama tik TRAIN sekų mediana.
    train_median = float(np.nanmedian(Xtr))
    X_filled = np.where(
        np.isnan(X_missing),
        train_median,
        X_missing
    )

    V_missing = torch.tensor(
        (X_filled - mu) / sd,
        dtype=torch.float32
    )

    with torch.no_grad():
        p_clean = (
            model(V_clean.to(DEVICE)).cpu().numpy() * ys + ym
        )
        p_missing = (
            model(V_missing.to(DEVICE)).cpu().numpy() * ys + ym
        )

    return p_clean, p_missing, float(mask.mean())


robust_rows = []

for fold_id, (tri, vai) in enumerate(folds, 1):
    tr = dev.iloc[tri]
    va = dev.iloc[vai]

    Xtr, ytr, _ = sequences(tr, 24)
    Xv, yv, _ = sequences(va, 24)

    p_clean, p_missing, actual_missing = fit_predict_clean_and_missing(
        Xtr, ytr, Xv,
        **best_n
    )

    # Piko slenkstis apskaičiuojamas tik iš konkretaus TRAIN laikotarpio.
    peak_threshold = float(
        tr["target"].quantile(PEAK_Q)
    )

    robust_rows.append({
        "fold": fold_id,
        "Sąlygos": "Įprasti duomenys",
        "Pašalinta_dalis": 0.0,
        **metrics(yv, p_clean, peak_threshold)
    })

    robust_rows.append({
        "fold": fold_id,
        "Sąlygos": "5 % trūkstamų reikšmių",
        "Pašalinta_dalis": actual_missing,
        **metrics(yv, p_missing, peak_threshold)
    })

robust_folds = pd.DataFrame(robust_rows)
display(robust_folds)

# Aiški ir fiksuota eilučių tvarka vėlesniems grafikams.
robust_table = (
    robust_folds
    .groupby("Sąlygos")[["MAE", "RMSE", "Peak_MAE", "PUR"]]
    .mean()
    .reindex(["Įprasti duomenys", "5 % trūkstamų reikšmių"])
)

display(robust_table)

plt.figure(figsize=(8, 4))
plt.bar(robust_table.index, robust_table["MAE"])
plt.ylabel("DEV MAE")
plt.title("N-BEATS atsparumas trūkstamoms įvesties reikšmėms")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

clean_mae = float(
    robust_table.loc["Įprasti duomenys", "MAE"]
)
missing_mae = float(
    robust_table.loc["5 % trūkstamų reikšmių", "MAE"]
)

change = (missing_mae - clean_mae) / clean_mae * 100

print(
    f"MAE pokytis dėl trūkstamų įvesčių: {change:+.2f} %"
)


### Išsamesnis atsparumo rezultatų palyginimas

Atsparumo poveikis vertinamas ne tik pagal MAE. Papildomi grafikai parodo, kaip 5 % trūkstamų įvesties reikšmių paveikia RMSE, piko valandų MAE ir PUR. Tai leidžia įvertinti, ar modelio pablogėjimas pasireiškia tik bendrame tikslume, ar ir piko valandomis.

In [ ]:
# MAE, RMSE ir Peak MAE turi tą patį vienetą.
ax = robust_table[["MAE", "RMSE", "Peak_MAE"]].plot(
    kind="bar",
    figsize=(10, 5)
)
ax.set_ylabel("Klaida, nuomų skaičius")
ax.set_title("Atsparumas: klaidų metrikų pokytis")
ax.set_xticklabels(
    ["Įprasti duomenys", "5 % trūkstamų reikšmių"],
    rotation=0
)
ax.legend(["MAE", "RMSE", "Peak MAE"])
plt.tight_layout()
plt.show()

# PUR rodomas atskirai, nes jo skalė yra 0–1.
plt.figure(figsize=(8, 4))
plt.bar(
    ["Įprasti duomenys", "5 % trūkstamų reikšmių"],
    robust_table["PUR"].to_numpy()
)
plt.ylabel("PUR")
plt.ylim(0, 1)
plt.title("Atsparumas: PUR pokytis")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 11. Mokymo kreivė ir prognozių analizė

Paskutiniame DEV skaidinyje parodoma užfiksuotos konfigūracijos mokymo eiga. `Early stopping` stebi vidinės chronologinės validavimo dalies nuostolį.

Toliau pateikiama faktinė ir prognozuota paklausa bei sklaidos diagrama. Ideali prognozė tenkintų:

$$
\hat y_i=y_i
$$

In [ ]:
# Vizualinei analizei naudojamas paskutinis DEV rolling-origin skaidinys.
# Jis nėra naudojamas naujiems hiperparametrams parinkti.
tri, vai = folds[-1]
tr, va = dev.iloc[tri], dev.iloc[vai]
Xtr, ytr, _ = sequences(tr, 24)
Xv, yv, tv = sequences(va, 24)

p, ep, hist = fit_predict(
    Xtr, ytr, Xv, **best_n,
    return_history=True
)

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(hist["train"])+1), hist["train"], label="Mokymo nuostolis")
plt.plot(range(1, len(hist["val"])+1), hist["val"], label="Validavimo nuostolis")
plt.xlabel("Epocha")
plt.ylabel("MAE standartizuotu masteliu")
plt.title("N-BEATS mokymo eiga")
plt.legend()
plt.tight_layout()
plt.show()

n = min(168, len(tv))
plt.figure(figsize=(13, 4))
plt.plot(tv[:n], yv[:n], label="Faktinė paklausa", linewidth=2)
plt.plot(tv[:n], p[:n], label="Prognozuota paklausa")
plt.xlabel("Laikas")
plt.ylabel("Nuomų skaičius")
plt.title("N-BEATS: faktinė ir prognozuota paklausa")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

lower = min(yv.min(), p.min())
upper = max(yv.max(), p.max())

plt.figure(figsize=(6, 6))
plt.scatter(yv, p, alpha=0.45)
plt.plot([lower, upper], [lower, upper], linestyle="--", label=r"Ideali prognozė: $\hat{y}=y$")
plt.xlabel("Faktinė paklausa")
plt.ylabel("Prognozuota paklausa")
plt.title("Faktinės ir prognozuotos paklausos palyginimas")
plt.legend()
plt.tight_layout()
plt.show()

print("Epochų skaičius:", ep)
display(pd.Series(metrics(
    yv, p,
    float(tr["target"].quantile(PEAK_Q))
)))

## 12. Piko valandų ir didžiausių klaidų analizė

Pasirašyta prognozavimo paklaida:

$$
e_i=y_i-\hat y_i
$$

Kai $e_i>0$, modelis paklausą **neįvertino**. Kai $e_i<0$, modelis paklausą **pervertino**.

Piko slenkstis skaičiuojamas tik iš atitinkamo TRAIN laikotarpio. Tai svarbu, nes piko apibrėžimui neturi būti naudojama informacija iš vertinamos ateities.

In [ ]:
# Piko slenkstis apskaičiuojamas tik iš TRAIN duomenų.
# Taip piko apibrėžimui nepanaudojama VALIDATION ateities informacija.
peak_threshold = float(tr["target"].quantile(PEAK_Q))
signed_error = yv - p
peak_mask = yv >= peak_threshold

plt.figure(figsize=(9, 4))
plt.hist(signed_error, bins=30)
plt.axvline(0, linestyle="--", label="Nulinė paklaida")
plt.xlabel(r"Paklaida $e_i=y_i-\hat{y}_i$")
plt.ylabel("Stebėjimų skaičius")
plt.title("N-BEATS prognozavimo paklaidų pasiskirstymas")
plt.legend()
plt.tight_layout()
plt.show()

err = pd.DataFrame({
    "Laikas": tv,
    "Faktinė paklausa": yv,
    "Prognozė": p,
    "Paklaida": signed_error,
    "Absoliuti klaida": np.abs(signed_error),
    "Piko valanda": peak_mask
})

print("10 didžiausių absoliučių klaidų:")
display(err.nlargest(10, "Absoliuti klaida"))

print("10 didžiausių klaidų piko valandomis:")
display(
    err.loc[err["Piko valanda"]]
       .nlargest(10, "Absoliuti klaida")
)

print(f"Piko slenkstis: {peak_threshold:.2f}")
print(f"Piko valandų skaičius: {int(peak_mask.sum())}")
print(f"Piko neįvertinimo dalis: {np.mean(p[peak_mask] < yv[peak_mask]):.3f}")

## 13. Praktinė reikšmė

Dviračių paklausos prognozė gali būti naudojama planuojant dviračių perskirstymą. Bendras MAE ir RMSE parodo prognozės tikslumą, tačiau operatoriui ypač svarbios piko valandos.

Didelė teigiama paklaida piko metu reiškia, kad tikroji paklausa buvo didesnė už prognozuotą. Tokiu atveju pagal prognozę suplanuotas dviračių kiekis galėtų būti per mažas. Todėl kartu su MAE ir RMSE vertinami **Peak MAE** ir **PUR**.

## 14. Galutinės N-BEATS konfigūracijos išsaugojimas

Galutiniam TEST palyginimui išsaugoma DEV dalyje pasirinkta konfigūracija. TEST rezultatai nėra naudojami šiai konfigūracijai keisti.

## Galutinė N-BEATS pasirinkimo logika

Galutinei N-BEATS konfigūracijai naudojamas **mažiausios vidutinės DEV MAE kriterijus**. Toks kriterijus taikomas prieš galutinį TEST ir todėl TEST duomenys nedaro įtakos modelio pasirinkimui.

PUR nėra naudojamas konfigūracijai parinkti. Jis paliekamas rezultatų lentelėse ir grafikuose kaip papildoma informacija apie modelio elgseną piko valandomis. Jei pagal MAE geresnis modelis turi didesnį PUR, tai interpretuojama kaip konkretus modelio ribotumas, o ne slepiama ar naudojama rezultatui dirbtinai koreguoti.

In [ ]:
# Galutiniam 06 Notebook išsaugoma tik DEV dalyje pasirinkta konfigūracija.
# Pagrindinis modelis lieka su 24 h langu; 24/168 h palyginimas šiame
# Notebook laikomas atskiru abliacijos eksperimentu, o ne TEST derinimu.
best_n_save = {
    **best_n,
    "lookback": 24,
    "selection_metric": "DEV_MAE",
    "model_type": "lightweight_NBEATS_type"
}

(RESULTS / "best_nbeats.json").write_text(
    json.dumps(best_n_save, indent=2),
    encoding="utf-8"
)

display(pd.DataFrame([best_n_save]).rename(columns={"hidden":"Paslėpto sluoksnio dydis", "blocks":"Blokų skaičius", "lr":"Mokymosi greitis", "lookback":"Retrospektyvus langas (val.)", "selection_metric": "DEV_MAE", "model_type":"Modelio tipas"}))

## Eksperimento metodikos santrauka

Eksperimentas vykdomas nuosekliai: paruošiamos ankstesnės paklausos sekos, įvertinamas pradinis N-BEATS tipo modelis, DEV dalyje atliekama hiperparametrų paieška ir pagal **mažiausią vidutinę DEV MAE** pasirenkama patobulinta konfigūracija.

RMSE ir Peak MAE naudojamos papildomam klaidų vertinimui. PUR pateikiamas kaip papildomas piko neįvertinimo diagnostinis rodiklis, tačiau modelio parinkimui nenaudojamas. Jei pagal MAE pasirinktas modelis pagal PUR veikia prasčiau, tai aiškiai nurodoma rezultatų interpretacijoje kaip modelio ribotumas.

Toliau tikrinamas rezultato stabilumas, atliekamas palyginimas su sezoniniu atskaitos metodu, 24/168 val. abliacija, 5 % trūkstamų reikšmių atsparumo bandymas ir klaidų analizė.

Visam modelio parinkimui naudojama tik DEV dalis. `rolling-origin` skaidiniuose mokymo laikotarpis visada yra ankstesnis už validavimo laikotarpį. Galutinis TEST šiame Notebook nenaudojamas modelio ar hiperparametrų pasirinkimui.

## 15. Ribotumai ir eksperimento išvada

Šiame darbe naudojama **lengva N-BEATS tipo realizacija**, o ne identiška oficialios N-BEATS bibliotekos architektūra. Modelio įvestis sudaryta tik iš ankstesnės paklausos sekos; meteorologiniai ir kalendoriniai požymiai tiesiogiai į šį modelį nepaduodami.

Hiperparametrų paieška yra ribota, todėl negalima teigti, kad rasta absoliučiai geriausia teoriškai įmanoma N-BEATS tipo konfigūracija. Pasirenkama geriausia iš šiame eksperimente patikrintų konfigūracijų pagal vidutinę DEV MAE.

Abliacijos bandymas tikrina 24 ir 168 valandų istorijos langus, o atsparumo bandymas – modelio elgseną, kai 5 % validavimo įvesties reikšmių trūksta. Abu bandymai atliekami DEV dalyje.

Galutinė išvada apie N-BEATS tipo modelio vietą tarp MLP, CatBoost ir optimizuoto Voting Ensemble bus daroma tik bendrame TEST rezultatų palyginime.